In [21]:
import pandas as pd
import numpy as np

In [52]:
df = pd.read_parquet("minable_view.parquet")

In [53]:
df

,id,track_name,artist_name,new_popularity_2025,acousticness,danceability,duration_ms,energy,explicit,instrumentalness,...,duration_cat,key_cat,imputed_date_cat,mode_cat,year_cat,genres_from_artists_cat,popularity_2025_cat,album,popularity_2024,popularity_2024_cat
0,0cS0A1fUEUd1EW3FcF8AEI,Keep A Song In Your Soul,Mamie Smith,18,0.991000,0.598,168333,0.224,0,0.000522,...,Mid,F,Yes,Minor,20s,Yes,FLOP,unknown,-1,unknown
1,11m7laMUgmOKqI3oYzuhne,Golfing Papa,Mamie Smith,1,0.993000,0.647,163827,0.186,0,0.000018,...,Mid,C,Yes,Major,20s,No,FLOP,unknown,-1,unknown
2,19Lc5SfJJ5O1oaxY0fpwfh,True House Music - Xavier Santos & Carlos Gomi...,Oscar Velazquez,13,0.000173,0.730,422087,0.798,0,0.801000,...,Large,D,No,Major,20s,Yes,FLOP,unknown,-1,unknown
3,2hJjbsLCytGsnAHfdsLejp,Xuniverxe,Mikesmooveee,0,0.295000,0.704,165224,0.707,1,0.000246,...,Mid,A♯/B♭,No,Minor,20s,Yes,FLOP,unknown,-1,unknown
4,3HnrHGLE9u2MjHtdobfWl9,Crazy Blues - 78rpm Version,Mamie Smith & Her Jazz Hounds,3,0.996000,0.424,198627,0.245,0,0.799000,...,Mid,F,Yes,Major,20s,No,FLOP,unknown,-1,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
280448,4NqQaovM14WR2hNPMFxgjb,25/8,Bad Bunny,70,0.092000,0.761,243276,0.732,0,0.000000,...,Mid,E,No,Minor,2020s,No,MID-HIT,unknown,70,MID-HIT
280449,3bkkMZEAhx7rTVz1C0itRQ,Black Swan,BTS,0,0.137000,0.719,198261,0.758,0,0.000000,...,Mid,D,No,Minor,2020s,No,FLOP,unknown,0,FLOP
280450,0IGxkEgMeF1dwOSIj2IUrT,Crisis,Jasiah,48,0.263000,0.760,109595,0.794,1,0.003240,...,Short,A♯/B♭,No,Minor,2020s,Yes,MID,unknown,52,MID-HIT
280451,5UusfWUMMLEXLMc1ViNZoe,@ MEH,Playboi Carti,59,0.013600,0.876,166799,0.492,1,0.000283,...,Mid,B,No,Minor,2020s,No,MID-HIT,unknown,60,MID-HIT


In [58]:
df.isna().any()

id                         False
track_name                  True
artist_name                 True
new_popularity_2025        False
acousticness               False
danceability               False
duration_ms                False
energy                     False
explicit                   False
instrumentalness           False
key                        False
liveness                   False
loudness                   False
mode                       False
popularity_2021            False
release_date               False
speechiness                False
tempo                      False
valence                    False
year                       False
genres                     False
main_genre                 False
subgenre_1                  True
subgenre_2                  True
genres_artists             False
acousticness_cat           False
danceability_cat           False
energy_cat                 False
explicit_cat               False
instrumentalness_cat       False
liveness_c

In [59]:
def season(month):
    if month in [12,1,2]:
        return 'Winter'
    elif month in [3,4,5]:
        return 'Spring'
    elif month in [6,7,8]:
        return 'Summer'
    else:
        return 'Autumn'

In [60]:
df["release_date"] = pd.to_datetime(
    df["release_date"], format="%Y-%m-%d", errors="coerce"
)

df["month"] = df["release_date"].dt.month
df["dayofweek"] = df["release_date"].dt.dayofweek

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df["is_weekend"] = df["dayofweek"].isin([5, 6])

df["season"] = df["month"].apply(season)

df['years_since_release'] = 2025 - df['year']

df['is_positive_mood'] = df['valence'] > 0.7

df['has_lyrics'] = df['speechiness'] > 0.33

df['is_loud'] = df['loudness'] > -6

df['is_live'] = df['liveness'] > 0.8

df['is_common_key'] = df['key'].isin([0,2,5,7])

df['is_instrumental'] = df['instrumentalness'] > 0.7

df['is_energetic'] = df['energy'] > 0.75

df['is_duration_friendly'] = df['duration_ms'].between(120000, 240000)

df['is_danceable'] = df['danceability'] > 0.6

df['is_acoustic'] = df['acousticness'] > 0.8

df['energy_valence'] = df['energy'] * df['valence']

df['speechiness_explicit'] = df['speechiness'] * df['explicit']

df['danceability_tempo'] = df['danceability'] * df['tempo']

df.drop(["release_date", "month", "dayofweek"], axis=1, inplace=True)

In [61]:
df.to_csv('minable_view_post.parquet', index=False)